<a href="https://colab.research.google.com/github/Ifaz2611/ML-Projects-By_Ifaz/blob/main/GalaxyPlot_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip -q install plotly

print ("Library Add Successfully")

Library Add Successfully


#The MIlky way

In [ ]:
# Advanced synthetic barred-spiral galaxy visualization for Google Colab
# -----------------------------------------------------------------------------
# This is a procedural visualization, not a physical N-body simulation.
# It adds reproducibility, configurable distributions, component-level styling,
# a simple rotation-velocity overlay, and interactive Plotly controls.

from dataclasses import dataclass
from typing import Dict, Tuple

import numpy as np
import plotly.graph_objects as go
from plotly.colors import sample_colorscale


# =============================================================================
# 1. Configuration
# =============================================================================

@dataclass
class GalaxyConfig:
    # Reproducibility and size
    seed: int = 42
    n_bulge: int = 18_000
    n_bar: int = 22_000
    n_disk: int = 75_000
    n_gas: int = 12_000
    n_halo: int = 8_000

    # Geometry in arbitrary galaxy-scale units (think kpc-like, but illustrative)
    bulge_scale: float = 0.75
    bulge_max_radius: float = 3.2
    bar_length: float = 6.5
    bar_width: float = 1.35
    bar_height: float = 0.65
    bar_angle_deg: float = 22.0
    disk_outer_radius: float = 17.0
    disk_scale_length: float = 4.2
    disk_scale_height: float = 0.34
    gas_scale_length: float = 5.5
    gas_scale_height: float = 0.12
    halo_radius: float = 28.0

    # Spiral-arm parameters
    n_arms: int = 4
    arm_pitch_deg: float = 13.0
    arm_fraction: float = 0.78
    arm_width: float = 0.17
    arm_winding: float = 1.0

    # Optional visual effects
    disk_warp: float = 0.10
    show_velocity_field: bool = True
    n_velocity_vectors: int = 420
    velocity_scale: float = 0.12
    sun_r: float = 8.2
    sun_phi_deg: float = 0.0
    sun_z: float = 0.0

    @property
    def bar_angle(self) -> float:
        return np.deg2rad(self.bar_angle_deg)

    @property
    def arm_pitch(self) -> float:
        return np.deg2rad(self.arm_pitch_deg)

    @property
    def sun_phi(self) -> float:
        return np.deg2rad(self.sun_phi_deg)


# =============================================================================
# 2. Sampling utilities
# =============================================================================

def rotate_xy(x: np.ndarray, y: np.ndarray, angle: float) -> Tuple[np.ndarray, np.ndarray]:
    """Rotate points counter-clockwise around the z-axis."""
    c, s = np.cos(angle), np.sin(angle)
    return c * x - s * y, s * x + c * y


def sample_isotropic_directions(rng: np.random.Generator, n: int):
    """Return n uniformly distributed directions on a sphere."""
    cos_theta = rng.uniform(-1.0, 1.0, n)
    sin_theta = np.sqrt(1.0 - cos_theta**2)
    phi = rng.uniform(0.0, 2.0 * np.pi, n)
    return sin_theta * np.cos(phi), sin_theta * np.sin(phi), cos_theta


def sample_bulge(cfg: GalaxyConfig, rng: np.random.Generator):
    """Sample a truncated Hernquist-like central bulge."""
    n = cfg.n_bulge
    # Hernquist CDF inversion, truncated to avoid a very long tail.
    u_max = (cfg.bulge_max_radius / (cfg.bulge_max_radius + cfg.bulge_scale)) ** 2
    u = rng.uniform(0.0, u_max, n)
    sqrt_u = np.sqrt(u)
    r = cfg.bulge_scale * sqrt_u / np.maximum(1.0 - sqrt_u, 1e-8)
    dx, dy, dz = sample_isotropic_directions(rng, n)
    return r * dx, r * dy, r * dz


def sample_bar(cfg: GalaxyConfig, rng: np.random.Generator):
    """Sample a smooth Ferrers-like triaxial bar inside an ellipsoid."""
    n = cfg.n_bar
    # Rejection sampling gives a denser centre than a uniform cuboid.
    x_parts, y_parts, z_parts = [], [], []
    collected = 0
    max_batch = max(2_000, n // 2)
    while collected < n:
        batch = max_batch
        x = rng.uniform(-cfg.bar_length / 2, cfg.bar_length / 2, batch)
        y = rng.uniform(-cfg.bar_width / 2, cfg.bar_width / 2, batch)
        z = rng.uniform(-cfg.bar_height / 2, cfg.bar_height / 2, batch)
        m2 = (2 * x / cfg.bar_length) ** 2 + (2 * y / cfg.bar_width) ** 2 + (2 * z / cfg.bar_height) ** 2
        inside = m2 < 1.0
        probability = np.where(inside, np.clip(1.0 - m2, 0.0, 1.0) ** 1.7, 0.0)
        keep = rng.random(batch) < probability
        if np.any(keep):
            x_parts.append(x[keep])
            y_parts.append(y[keep])
            z_parts.append(z[keep])
            collected += int(keep.sum())

    x = np.concatenate(x_parts)[:n]
    y = np.concatenate(y_parts)[:n]
    z = np.concatenate(z_parts)[:n]
    x, y = rotate_xy(x, y, cfg.bar_angle)
    return x, y, z


def sample_spiral_component(
    cfg: GalaxyConfig,
    rng: np.random.Generator,
    n: int,
    scale_length: float,
    scale_height: float,
    arm_scatter: float,
    arm_fraction: float,
    is_gas: bool = False,
):
    """Sample an exponential disk with logarithmic spiral-arm overdensities."""
    # A 2-D exponential disk has Gamma(k=2) radial distribution.
    r = rng.gamma(shape=2.0, scale=scale_length, size=n)
    r = np.clip(r, 0.08, cfg.disk_outer_radius)
    phi_random = rng.uniform(0.0, 2.0 * np.pi, n)

    arm_ids = rng.integers(0, cfg.n_arms, size=n)
    arm_base = 2.0 * np.pi * arm_ids / cfg.n_arms
    reference_radius = 1.0
    spiral_phase = cfg.arm_winding * np.log(r / reference_radius) / np.tan(cfg.arm_pitch)
    phi_arm = arm_base + spiral_phase

    in_arm = rng.random(n) < arm_fraction
    phi = np.where(
        in_arm,
        phi_arm + rng.normal(0.0, arm_scatter * (0.55 + r / cfg.disk_outer_radius), n),
        phi_random,
    )

    x = r * np.cos(phi)
    y = r * np.sin(phi)

    # A mild warp starts in the outer disk and preserves a thin central plane.
    warp = cfg.disk_warp * np.maximum(r - 0.55 * cfg.disk_outer_radius, 0.0) ** 2
    warp /= max((0.45 * cfg.disk_outer_radius) ** 2, 1e-8)
    z = rng.normal(0.0, scale_height * (0.8 + 0.45 * r / cfg.disk_outer_radius), n)
    z += warp * np.sin(phi - 0.35)

    # Young gas traces arms more tightly and receives a small clump offset.
    if is_gas:
        z += rng.normal(0.0, 0.025, n)
        x += rng.normal(0.0, 0.035, n)
        y += rng.normal(0.0, 0.035, n)

    return x, y, z, r, phi, in_arm


def sample_halo(cfg: GalaxyConfig, rng: np.random.Generator):
    """Sample a faint, diffuse stellar halo with a softened power-law profile."""
    n = cfg.n_halo
    u = rng.uniform(0.0, 1.0, n)
    # Concentrate points toward the centre while retaining an extended halo.
    r = cfg.halo_radius * u ** 0.58
    dx, dy, dz = sample_isotropic_directions(rng, n)
    # Slight flattening makes the halo visually less spherical and more galactic.
    return r * dx, r * dy, 0.72 * r * dz


def rotation_speed(r: np.ndarray, vmax: float = 1.0, turnover: float = 2.8) -> np.ndarray:
    """Simple rising-then-flat rotation curve used only for visual context."""
    return vmax * (1.0 - np.exp(-r / turnover))


# =============================================================================
# 3. Galaxy generation
# =============================================================================

def generate_galaxy(cfg: GalaxyConfig) -> Dict[str, Dict[str, np.ndarray]]:
    rng = np.random.default_rng(cfg.seed)

    xb, yb, zb = sample_bulge(cfg, rng)
    xbar, ybar, zbar = sample_bar(cfg, rng)
    xd, yd, zd, rd, phid, arm_mask = sample_spiral_component(
        cfg, rng, cfg.n_disk, cfg.disk_scale_length, cfg.disk_scale_height,
        cfg.arm_width, cfg.arm_fraction,
    )
    xg, yg, zg, rg, phig, gas_arm_mask = sample_spiral_component(
        cfg, rng, cfg.n_gas, cfg.gas_scale_length, cfg.gas_scale_height,
        cfg.arm_width * 0.48, min(cfg.arm_fraction + 0.12, 0.98), is_gas=True,
    )
    xh, yh, zh = sample_halo(cfg, rng)

    return {
        "halo": {"x": xh, "y": yh, "z": zh},
        "bulge": {"x": xb, "y": yb, "z": zb},
        "bar": {"x": xbar, "y": ybar, "z": zbar},
        "disk": {"x": xd, "y": yd, "z": zd, "r": rd, "phi": phid, "arm": arm_mask},
        "gas": {"x": xg, "y": yg, "z": zg, "r": rg, "phi": phig, "arm": gas_arm_mask},
    }


# =============================================================================
# 4. Plotly visualization
# =============================================================================

COMPONENT_STYLE = {
    "halo":  dict(color="#7586a6", size=1.2, opacity=0.12, label="Stellar halo"),
    "bulge": dict(color="#ffcf75", size=2.5, opacity=0.34, label="Bulge"),
    "bar":   dict(color="#ff9f5a", size=2.3, opacity=0.52, label="Central bar"),
    "disk":  dict(color="#8bc8ff", size=1.7, opacity=0.50, label="Spiral disk"),
    "gas":   dict(color="#b9f3ff", size=2.0, opacity=0.65, label="Gas / star-forming arms"),
}


def make_scatter_trace(name: str, values: Dict[str, np.ndarray]) -> go.Scatter3d:
    style = COMPONENT_STYLE[name]
    marker = dict(size=style["size"], color=style["color"], opacity=style["opacity"])

    # Highlight arm-associated disk particles with a subtle size difference.
    if name in {"disk", "gas"} and "arm" in values:
        marker["size"] = np.where(values["arm"], style["size"] * 1.25, style["size"] * 0.72)

    return go.Scatter3d(
        x=values["x"], y=values["y"], z=values["z"],
        mode="markers", name=style["label"], marker=marker,
        hovertemplate=(
            f"<b>{style['label']}</b><br>"
            "x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>"
        ),
    )


def make_velocity_cones(cfg: GalaxyConfig, galaxy: Dict[str, Dict[str, np.ndarray]], rng):
    if not cfg.show_velocity_field:
        return None

    disk = galaxy["disk"]
    n = min(cfg.n_velocity_vectors, len(disk["x"]))
    ids = rng.choice(len(disk["x"]), size=n, replace=False)
    x, y, z = disk["x"][ids], disk["y"][ids], disk["z"][ids]
    r = np.hypot(x, y)
    speed = rotation_speed(r)
    phi = np.arctan2(y, x)
    u = -np.sin(phi) * speed * cfg.velocity_scale
    v = np.cos(phi) * speed * cfg.velocity_scale
    w = rng.normal(0.0, 0.012, n)

    return go.Cone(
        x=x, y=y, z=z, u=u, v=v, w=w,
        sizemode="scaled", sizeref=0.55, anchor="tail",
        colorscale=[[0.0, "#6ee7ff"], [1.0, "#ffffff"]],
        showscale=False, opacity=0.48, name="Rotation field",
        hoverinfo="skip",
    )


def make_galaxy_figure(cfg: GalaxyConfig, galaxy: Dict[str, Dict[str, np.ndarray]]) -> go.Figure:
    traces = [make_scatter_trace(name, galaxy[name]) for name in COMPONENT_STYLE]

    sun_x = cfg.sun_r * np.cos(cfg.sun_phi)
    sun_y = cfg.sun_r * np.sin(cfg.sun_phi)
    sun_trace = go.Scatter3d(
        x=[sun_x], y=[sun_y], z=[cfg.sun_z], mode="markers+text",
        text=["Sun"], textposition="top center", name="Sun (approx.)",
        marker=dict(size=7, color="#fff200", symbol="diamond", line=dict(color="white", width=1)),
        hovertemplate="<b>Sun (approx.)</b><br>r=%{x:.2f}<extra></extra>",
    )
    traces.append(sun_trace)

    velocity_trace = make_velocity_cones(cfg, galaxy, np.random.default_rng(cfg.seed + 1))
    if velocity_trace is not None:
        traces.append(velocity_trace)

    fig = go.Figure(data=traces)

    # Visibility presets: all, stellar structure, and disk/arms only.
    n = len(traces)
    all_visible = [True] * n
    stellar_visible = [True, True, True, True, False, True] + ([False] if n == 7 else [])
    disk_visible = [False, False, False, True, True, True] + ([False] if n == 7 else [])
    buttons = [
        dict(label="All components", method="update", args=[{"visible": all_visible}, {"title": "Synthetic Barred Spiral Galaxy — all components"}]),
        dict(label="Stellar structure", method="update", args=[{"visible": stellar_visible}, {"title": "Synthetic Barred Spiral Galaxy — stellar structure"}]),
        dict(label="Disk and arms", method="update", args=[{"visible": disk_visible}, {"title": "Synthetic Barred Spiral Galaxy — disk and arms"}]),
    ]

    fig.update_layout(
        title="Synthetic Barred Spiral Galaxy — all components",
        template="plotly_dark",
        paper_bgcolor="#050814",
        plot_bgcolor="#050814",
        font=dict(family="Arial, sans-serif", color="#e7eefc"),
        scene=dict(
            xaxis=dict(title="X [galaxy units]", showbackground=False, gridcolor="#1b2940"),
            yaxis=dict(title="Y [galaxy units]", showbackground=False, gridcolor="#1b2940"),
            zaxis=dict(title="Z [galaxy units]", showbackground=False, gridcolor="#1b2940"),
            aspectmode="data",
            camera=dict(eye=dict(x=1.55, y=1.55, z=0.95)),
        ),
        updatemenus=[dict(
            type="dropdown", direction="down", x=0.01, y=0.99,
            xanchor="left", yanchor="top", buttons=buttons,
            bgcolor="#101b31", bordercolor="#5272a5",
        )],
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0),
        margin=dict(l=0, r=0, b=0, t=70),
        uirevision="galaxy-layout",
    )
    return fig


def make_top_down_figure(cfg: GalaxyConfig, galaxy: Dict[str, Dict[str, np.ndarray]]) -> go.Figure:
    """Create a lighter 2-D top-down companion view for fast inspection."""
    fig = go.Figure()
    for name in COMPONENT_STYLE:
        values = galaxy[name]
        style = COMPONENT_STYLE[name]
        marker_size = style["size"] * (1.5 if name in {"bar", "gas"} else 1.0)
        fig.add_trace(go.Scattergl(
            x=values["x"], y=values["y"], mode="markers", name=style["label"],
            marker=dict(size=marker_size, color=style["color"], opacity=min(style["opacity"] + 0.12, 0.85)),
            hovertemplate=f"<b>{style['label']}</b><br>x=%{{x:.2f}}<br>y=%{{y:.2f}}<extra></extra>",
        ))

    fig.add_trace(go.Scatter(
        x=[cfg.sun_r * np.cos(cfg.sun_phi)], y=[cfg.sun_r * np.sin(cfg.sun_phi)],
        mode="markers+text", text=["Sun"], textposition="top center", name="Sun (approx.)",
        marker=dict(size=10, color="#fff200", symbol="diamond"),
    ))
    fig.update_layout(
        title="Top-down projection",
        template="plotly_dark", paper_bgcolor="#050814", plot_bgcolor="#050814",
        xaxis=dict(title="X [galaxy units]", scaleanchor="y", scaleratio=1, gridcolor="#1b2940"),
        yaxis=dict(title="Y [galaxy units]", gridcolor="#1b2940"),
        margin=dict(l=40, r=20, b=45, t=55),
    )
    return fig


# =============================================================================
# 5. Run in Google Colab
# =============================================================================

cfg = GalaxyConfig(
    seed=42,
    n_bulge=18_000,
    n_bar=22_000,
    n_disk=75_000,
    n_gas=12_000,
    n_halo=8_000,
    n_arms=4,
    arm_pitch_deg=13.0,
    bar_angle_deg=22.0,
    show_velocity_field=True,
)

galaxy = generate_galaxy(cfg)
fig_3d = make_galaxy_figure(cfg, galaxy)
fig_3d.show()

# The 2-D view is useful for checking the spiral structure without perspective.
fig_top_down = make_top_down_figure(cfg, galaxy)
fig_top_down.show()

print(
    f"Generated {sum(len(part['x']) for part in galaxy.values()):,} particles "
    f"with seed={cfg.seed}."
)

# Optional exports from Colab:
# fig_3d.write_html("synthetic_barred_spiral_galaxy_3d.html")
# fig_top_down.write_html("synthetic_barred_spiral_galaxy_top_down.html")
# fig_3d.write_image("synthetic_barred_spiral_galaxy.png", scale=2)

# Optional parameter experiments:
# cfg.arm_pitch_deg = 20.0       # more open arms
# cfg.n_arms = 2                  # grand-design two-arm galaxy
# cfg.bar_angle_deg = 45.0        # rotate the central bar
# cfg.arm_fraction = 0.90         # stronger arms
# cfg.show_velocity_field = False # faster rendering on low-memory sessions
# galaxy = generate_galaxy(cfg)
# make_galaxy_figure(cfg, galaxy).show()
# make_top_down_figure(cfg, galaxy).show()


__all__ = [
    "GalaxyConfig", "generate_galaxy", "make_galaxy_figure",
    "make_top_down_figure", "rotation_speed",
]


if __name__ == "__main__":
    # Running this file outside Colab produces the same interactive figures.
    print("Run the final section in a notebook cell, or import the functions above.")

# End of notebook-ready script

# Upgrade summary:
# - Dataclass configuration replaces scattered global parameters.
# - Local NumPy Generator gives reproducible, independent random streams.
# - Bulge uses a Hernquist-like profile; bar uses Ferrers-like rejection sampling.
# - Disk and gas use exponential radial profiles plus logarithmic spiral arms.
# - Adds a diffuse stellar halo, disk warp, gas clumping, and a rotation field.
# - Adds 3-D and top-down views with component visibility controls.
# - Uses Scattergl in the 2-D view for faster rendering of large point clouds.
# - Includes export hooks for HTML and high-resolution images.
# - Keeps all assumptions explicit: this is an illustrative procedural model.


#Colaiding MIlky way and Andromeda.

In [ ]:
# Animated Milky Way–Andromeda collision visualization for Google Colab
# -----------------------------------------------------------------------------
# This is an illustrative procedural animation. It is not a numerical gravity
# simulation and does not predict the exact future orbit of either galaxy.

from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import plotly.graph_objects as go


# =============================================================================
# 1. Configuration
# =============================================================================

@dataclass
class CollisionConfig:
    seed: int = 42
    n_stars_each: int = 4_500
    n_frames: int = 42
    animation_duration_ms: int = 100

    # Initial separation and encounter geometry.
    initial_separation: float = 28.0
    closest_approach: float = 2.8
    final_remnant_radius: float = 18.0
    orbital_tilt_deg: float = 18.0
    encounter_angle_deg: float = 25.0

    # Visual geometry.
    disk_radius: float = 11.0
    disk_scale_length: float = 3.5
    disk_thickness: float = 0.28
    arm_pitch_deg: float = 14.0
    n_arms: int = 4
    arm_strength: float = 0.82
    tail_strength: float = 1.0
    tail_spread: float = 0.65

    # Plotly rendering.
    point_size: float = 2.1
    show_labels: bool = True

    @property
    def orbital_tilt(self) -> float:
        return np.deg2rad(self.orbital_tilt_deg)

    @property
    def encounter_angle(self) -> float:
        return np.deg2rad(self.encounter_angle_deg)

    @property
    def arm_pitch(self) -> float:
        return np.deg2rad(self.arm_pitch_deg)


# =============================================================================
# 2. Galaxy sampling helpers
# =============================================================================

def rotate_xyz(x, y, z, angle_z=0.0, angle_x=0.0):
    """Rotate points around z, then around x."""
    cz, sz = np.cos(angle_z), np.sin(angle_z)
    x1 = cz * x - sz * y
    y1 = sz * x + cz * y
    z1 = z

    cx, sx = np.cos(angle_x), np.sin(angle_x)
    y2 = cx * y1 - sx * z1
    z2 = sx * y1 + cx * z1
    return x1, y2, z2


def sample_galaxy(cfg: CollisionConfig, rng: np.random.Generator, seed_offset: int = 0) -> Dict[str, np.ndarray]:
    """Create one stylized barred spiral with a disk, bulge, and two tail seeds."""
    local_rng = np.random.default_rng(cfg.seed + seed_offset)
    n = cfg.n_stars_each

    # Exponential disk radial distribution.
    r = local_rng.gamma(shape=2.0, scale=cfg.disk_scale_length, size=n)
    r = np.clip(r, 0.12, cfg.disk_radius)
    phi0 = local_rng.uniform(0.0, 2.0 * np.pi, n)

    # Logarithmic spiral arm overdensity.
    arm_id = local_rng.integers(0, cfg.n_arms, n)
    arm_base = 2.0 * np.pi * arm_id / cfg.n_arms
    spiral_phase = np.log(r / 1.0) / np.tan(cfg.arm_pitch)
    phi_arm = arm_base + spiral_phase
    in_arm = local_rng.random(n) < cfg.arm_strength
    phi = np.where(
        in_arm,
        phi_arm + local_rng.normal(0.0, 0.18 + 0.015 * r, n),
        phi0,
    )

    x = r * np.cos(phi)
    y = r * np.sin(phi)
    z = local_rng.normal(0.0, cfg.disk_thickness * (0.7 + 0.3 * r / cfg.disk_radius), n)

    # Central bulge particles are mixed into the same trace for speed.
    n_bulge = max(1, n // 6)
    rb = 2.0 * local_rng.random(n_bulge) ** 0.45
    phib = local_rng.uniform(0.0, 2.0 * np.pi, n_bulge)
    xb = rb * np.cos(phib)
    yb = rb * np.sin(phib)
    zb = local_rng.normal(0.0, 0.45, n_bulge)

    x = np.concatenate([x, xb])
    y = np.concatenate([y, yb])
    z = np.concatenate([z, zb])
    r = np.concatenate([r, rb])
    phi_all = np.concatenate([phi, phib])

    # Rotate each galaxy internally so their disks are not identical.
    internal_z = np.deg2rad(14.0 if seed_offset == 0 else -24.0)
    internal_x = np.deg2rad(8.0 if seed_offset == 0 else -12.0)
    x, y, z = rotate_xyz(x, y, z, internal_z, internal_x)

    # A subset of outer stars becomes the seed for tidal tails.
    tail_seed = r > np.quantile(r, 0.70)
    tail_sign = np.where(np.sin(phi_all + seed_offset) >= 0.0, 1.0, -1.0)
    tail_direction = tail_sign * (0.75 + 0.25 * local_rng.random(len(tail_seed)))

    return {
        "x": x,
        "y": y,
        "z": z,
        "r": r,
        "tail_seed": tail_seed,
        "tail_direction": tail_direction,
        "arm": np.concatenate([in_arm, np.ones(n_bulge, dtype=bool)]),
    }


def smoothstep(x: np.ndarray) -> np.ndarray:
    return x * x * (3.0 - 2.0 * x)


def encounter_centers(cfg: CollisionConfig, t: float) -> Tuple[np.ndarray, np.ndarray]:
    """Return stylized galaxy centers for normalized time t in [0, 1]."""
    # Approach from opposing sides, pass near one another, then form one remnant.
    approach = 1.0 - smoothstep(np.clip(t / 0.52, 0.0, 1.0))
    separation = cfg.closest_approach + (cfg.initial_separation - cfg.closest_approach) * approach

    # A gentle transverse offset gives the encounter a curved fly-by appearance.
    transverse = 3.2 * np.sin(np.pi * np.clip(t, 0.0, 1.0))
    remnant_mix = smoothstep(np.clip((t - 0.55) / 0.45, 0.0, 1.0))

    c1_pre = np.array([-separation / 2.0, -transverse / 2.0, 0.0])
    c2_pre = np.array([ separation / 2.0,  transverse / 2.0, 0.0])
    c1_post = np.array([-0.65 * (1.0 - remnant_mix), 0.5 * (1.0 - remnant_mix), 0.0])
    c2_post = np.array([ 0.65 * (1.0 - remnant_mix), -0.5 * (1.0 - remnant_mix), 0.0])

    return (1.0 - remnant_mix) * c1_pre + remnant_mix * c1_post, (1.0 - remnant_mix) * c2_pre + remnant_mix * c2_post


def transform_galaxy_for_time(
    cfg: CollisionConfig,
    base: Dict[str, np.ndarray],
    t: float,
    galaxy_index: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Move, rotate, stretch, and tail-distort one galaxy at time t."""
    c1, c2 = encounter_centers(cfg, t)
    center = c1 if galaxy_index == 0 else c2
    sign = -1.0 if galaxy_index == 0 else 1.0

    # The disks rotate as the galaxies approach and become more disturbed later.
    rotation = sign * (0.25 + 1.55 * t)
    x, y, z = rotate_xyz(base["x"], base["y"], base["z"], rotation, 0.0)

    interaction = smoothstep(np.clip((t - 0.20) / 0.58, 0.0, 1.0))
    r = np.hypot(x, y)
    phi = np.arctan2(y, x)

    # Tidal stretching and an S-shaped vertical disturbance.
    stretch = 1.0 + 0.55 * interaction * (r / cfg.disk_radius) ** 1.5
    x = x * (1.0 + 0.16 * interaction) + sign * 0.18 * interaction * y
    y = y * stretch
    z = z + 0.42 * interaction * (r / cfg.disk_radius) ** 1.6 * np.sin(phi + sign * 1.3)

    # Add an extended tail using the outer-disk seed particles.
    tail = base["tail_seed"]
    tail_progress = np.clip((t - 0.18) / 0.58, 0.0, 1.0)
    tail_progress = smoothstep(tail_progress)
    tail_axis = sign * base["tail_direction"] * cfg.tail_strength * 11.0 * tail_progress
    x = x + tail * tail_axis * np.cos(cfg.encounter_angle)
    y = y + tail * tail_axis * np.sin(cfg.encounter_angle)
    y = y + tail * sign * 1.1 * interaction * np.sin(phi * 1.7)
    z = z + tail * 0.55 * interaction * np.cos(phi)

    # A small common spread appears during the final merger stage.
    merger = smoothstep(np.clip((t - 0.60) / 0.40, 0.0, 1.0))
    x = x * (1.0 - 0.18 * merger) + np.sin(phi * 2.0 + galaxy_index) * merger * 0.35 * r / cfg.disk_radius
    y = y * (1.0 - 0.18 * merger) + np.cos(phi * 2.0 + galaxy_index) * merger * 0.35 * r / cfg.disk_radius
    z = z * (1.0 - 0.10 * merger)

    # Opacity is used to transition from two visible disks to a remnant trace.
    galaxy_opacity = max(0.0, 1.0 - merger)
    return x + center[0], y + center[1], z + center[2], np.full(len(x), galaxy_opacity)


def make_merged_remnant(cfg: CollisionConfig, galaxy_a, galaxy_b, t: float):
    """Create the final warm, diffuse merged remnant from both galaxies."""
    x = np.concatenate([galaxy_a["x"], galaxy_b["x"]])
    y = np.concatenate([galaxy_a["y"], galaxy_b["y"]])
    z = np.concatenate([galaxy_a["z"], galaxy_b["z"]])
    rng = np.random.default_rng(cfg.seed + 900)

    r = np.hypot(x, y)
    phi = np.arctan2(y, x)
    final_mix = smoothstep(np.clip((t - 0.58) / 0.42, 0.0, 1.0))
    x = x * (0.94 + 0.06 * rng.random(len(x)))
    y = y * (0.94 + 0.06 * rng.random(len(y)))
    z = z * 1.25 + rng.normal(0.0, 0.08 + 0.12 * final_mix, len(z))

    # Broad tidal debris remains visible around the remnant.
    debris = (r > cfg.disk_radius * 0.55).astype(float)
    x += debris * 3.8 * final_mix * np.cos(phi + 0.40)
    y += debris * 3.8 * final_mix * np.sin(phi + 0.40)
    return x, y, z


# =============================================================================
# 3. Animated Plotly figure
# =============================================================================

def phase_label(t: float) -> str:
    if t < 0.22:
        return "Approach"
    if t < 0.55:
        return "First encounter — tidal tails forming"
    if t < 0.78:
        return "Merger — disks losing their identity"
    return "Merged remnant — diffuse tidal debris remains"


def make_collision_figure(cfg: CollisionConfig) -> go.Figure:
    rng = np.random.default_rng(cfg.seed)
    mw = sample_galaxy(cfg, rng, seed_offset=0)
    andromeda = sample_galaxy(cfg, rng, seed_offset=101)

    # Trace order is fixed so animation frames can update efficiently.
    initial_mw = transform_galaxy_for_time(cfg, mw, 0.0, 0)
    initial_m31 = transform_galaxy_for_time(cfg, andromeda, 0.0, 1)
    remnant_x, remnant_y, remnant_z = make_merged_remnant(cfg, mw, andromeda, 0.0)

    traces = [
        go.Scatter3d(
            x=initial_mw[0], y=initial_mw[1], z=initial_mw[2],
            mode="markers", name="Milky Way",
            marker=dict(size=cfg.point_size, color="#6eb6ff", opacity=0.95),
            hoverinfo="skip",
        ),
        go.Scatter3d(
            x=initial_m31[0], y=initial_m31[1], z=initial_m31[2],
            mode="markers", name="Andromeda (M31)",
            marker=dict(size=cfg.point_size, color="#ff9d6e", opacity=0.95),
            hoverinfo="skip",
        ),
        go.Scatter3d(
            x=remnant_x, y=remnant_y, z=remnant_z,
            mode="markers", name="Merged remnant",
            marker=dict(size=cfg.point_size * 1.08, color="#ffd28a", opacity=0.0),
            hoverinfo="skip",
        ),
        go.Scatter3d(
            x=[initial_mw[0].mean(), initial_m31[0].mean()],
            y=[initial_mw[1].mean(), initial_m31[1].mean()],
            z=[initial_mw[2].mean() + 2.4, initial_m31[2].mean() + 2.4],
            mode="text", text=["Milky Way", "Andromeda"],
            textfont=dict(size=13, color=["#6eb6ff", "#ff9d6e"]),
            name="Galaxy labels", showlegend=False,
        ),
    ]

    frames: List[go.Frame] = []
    for frame_id, t in enumerate(np.linspace(0.0, 1.0, cfg.n_frames)):
        mw_frame = transform_galaxy_for_time(cfg, mw, t, 0)
        m31_frame = transform_galaxy_for_time(cfg, andromeda, t, 1)
        rem_x, rem_y, rem_z = make_merged_remnant(cfg, mw, andromeda, t)
        c1, c2 = encounter_centers(cfg, t)

        merger = smoothstep(np.clip((t - 0.58) / 0.42, 0.0, 1.0))
        body_opacity = max(0.0, 1.0 - merger)
        remnant_opacity = merger
        label_text = ["Milky Way", "Andromeda"] if t < 0.73 else ["", ""]

        frames.append(go.Frame(
            name=f"frame_{frame_id:03d}",
            data=[
                go.Scatter3d(
                    x=mw_frame[0], y=mw_frame[1], z=mw_frame[2],
                    marker=dict(opacity=body_opacity),
                ),
                go.Scatter3d(
                    x=m31_frame[0], y=m31_frame[1], z=m31_frame[2],
                    marker=dict(opacity=body_opacity),
                ),
                go.Scatter3d(
                    x=rem_x, y=rem_y, z=rem_z,
                    marker=dict(opacity=remnant_opacity),
                ),
                go.Scatter3d(
                    x=[c1[0], c2[0]], y=[c1[1], c2[1]], z=[c1[2] + 2.4, c2[2] + 2.4],
                    text=label_text,
                ),
            ],
            layout=dict(title=f"Milky Way × Andromeda — {phase_label(t)} | t={t:.2f}"),
        ))

    play_button = dict(
        label="Play collision", method="animate",
        args=[None, dict(frame=dict(duration=cfg.animation_duration_ms, redraw=False),
                          transition=dict(duration=cfg.animation_duration_ms // 2),
                          fromcurrent=True, mode="immediate")],
    )
    pause_button = dict(
        label="Pause", method="animate",
        args=[[None], dict(frame=dict(duration=0, redraw=False), transition=dict(duration=0), mode="immediate")],
    )

    fig = go.Figure(data=traces, frames=frames)
    fig.update_layout(
        title=f"Milky Way × Andromeda — {phase_label(0.0)} | t=0.00",
        template="plotly_dark",
        paper_bgcolor="#03050d",
        plot_bgcolor="#03050d",
        font=dict(family="Arial, sans-serif", color="#eaf2ff"),
        scene=dict(
            xaxis=dict(title="X [galaxy units]", showbackground=False, gridcolor="#19243c"),
            yaxis=dict(title="Y [galaxy units]", showbackground=False, gridcolor="#19243c"),
            zaxis=dict(title="Z [galaxy units]", showbackground=False, gridcolor="#19243c"),
            aspectmode="data",
            camera=dict(eye=dict(x=1.45, y=1.45, z=0.86)),
        ),
        updatemenus=[dict(
            type="buttons", direction="left", x=0.02, y=0.98,
            xanchor="left", yanchor="top", buttons=[play_button, pause_button],
            bgcolor="#111b30", bordercolor="#4e6b9f",
        )],
        sliders=[dict(
            active=0, x=0.12, y=0.045, len=0.82,
            currentvalue=dict(prefix="Encounter time: ", visible=True),
            steps=[dict(
                label=f"{t:.2f}", method="animate", args=[[f"frame_{i:03d}"],
                dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))]
            ) for i, t in enumerate(np.linspace(0.0, 1.0, cfg.n_frames))],
        )],
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0),
        margin=dict(l=0, r=0, b=0, t=75),
        uirevision="collision-layout",
    )
    return fig


# =============================================================================
# 4. Run in Google Colab
# =============================================================================

cfg = CollisionConfig(
    seed=42,
    n_stars_each=4_500,
    n_frames=42,
    initial_separation=28.0,
    closest_approach=2.8,
    orbital_tilt_deg=18.0,
    encounter_angle_deg=25.0,
)

collision_fig = make_collision_figure(cfg)
collision_fig.show()

# Optional export:
# collision_fig.write_html("milky_way_andromeda_collision.html")
# For a static image, install Kaleido first:
# !pip -q install kaleido
# collision_fig.write_image("milky_way_andromeda_collision.png", scale=2)

# Performance tuning for Colab:
# cfg.n_stars_each = 2_000
# cfg.n_frames = 25
# collision_fig = make_collision_figure(cfg)
# collision_fig.show()

__all__ = ["CollisionConfig", "sample_galaxy", "make_collision_figure"]
